# 12. Qualitative Translation Error & Synonym Analysis
Categorizes translation errors on example predictions -- run against real model output once available (notebook 09); the categorization logic itself is demonstrated here on illustrative examples.

In [1]:
# ============================================================
# PATH BOOSTER -- guarantees project root in sys.path & CWD.
# Matches notebooks/00_setup_environment.ipynb's convention: this repo
# is deployed both to Colab (fresh `git clone`, folder "Ekegusii-LLM-Translation")
# and to Kineses Cloud / similar Jupyter hosts (pre-placed at
# ~/Ekegusii-LLM-Translation-main -- the "-main" suffix comes from GitHub's
# "Download ZIP" naming). Do not assume either folder name is the cwd.
# ============================================================
import os
import sys

REPO_NAME = "Ekegusii-LLM-Translation"


def _find_project_root():
    if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
        if not os.path.exists(REPO_NAME):
            os.system(f"git clone https://github.com/aykahsay/{REPO_NAME}.git")
            os.system(f"pip install -q -r {REPO_NAME}/requirements.txt")
        return os.path.abspath(REPO_NAME)

    try:
        cwd = os.getcwd()
    except FileNotFoundError:
        cwd = os.path.expanduser("~")
        os.chdir(cwd)

    home = os.path.expanduser("~")
    for candidate in (f"{REPO_NAME}-main", REPO_NAME):
        proj_dir = os.path.join(home, candidate)
        if os.path.isdir(proj_dir):
            return proj_dir

    if os.path.exists("src") and os.path.exists("data"):
        return cwd
    if os.path.basename(cwd) == "notebooks" and os.path.exists(os.path.join("..", "src")):
        return os.path.abspath("..")

    return cwd


project_root = _find_project_root()
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Project root: {project_root}")


Project root: C:\Users\Admin\OneDrive - United States International University (USIU)\Documents\NLP\Multilogual_transaltion_nlp


In [2]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


numpy/pandas ABI OK.


In [3]:
import pandas as pd

# Illustrative (source, reference, hypothesis) triples -- replace with
# real predictions saved from notebook 09 for genuine error analysis.
examples = pd.DataFrame({
    'source': [
        'Wash your hands frequently with soap.',
        'The Ministry of Health announced a new vaccination campaign.',
        'Farmers should plant drought-resistant crops.',
    ],
    'reference': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu mocha ogotema.',
        'Abasaki bagoika gotema amakoro agatangete oborwa amanche.',
    ],
    'hypothesis': [
        'Esibie amaboko ao botambe na esabuni.',
        'Ewizara ya obochenu yatangaza omochenu.',
        'Abasaki bagoika gotema.',
    ],
})
examples

,source,reference,hypothesis
0,Wash your hands frequently with soap.,Esibie amaboko ao botambe na esabuni.,Esibie amaboko ao botambe na esabuni.
1,The Ministry of Health announced a new vaccina...,Ewizara ya obochenu yatangaza omochenu mocha o...,Ewizara ya obochenu yatangaza omochenu.
2,Farmers should plant drought-resistant crops.,Abasaki bagoika gotema amakoro agatangete obor...,Abasaki bagoika gotema.


## Length-ratio error flag (a cheap proxy for omission/truncation)

In [4]:
def word_count(text):
    return len(str(text).split())

examples['ref_len'] = examples['reference'].apply(word_count)
examples['hyp_len'] = examples['hypothesis'].apply(word_count)
examples['length_ratio'] = examples['hyp_len'] / examples['ref_len']
examples['likely_omission'] = examples['length_ratio'] < 0.7
examples[['source', 'ref_len', 'hyp_len', 'length_ratio', 'likely_omission']]

,source,ref_len,hyp_len,length_ratio,likely_omission
0,Wash your hands frequently with soap.,6,6,1.000000,False
1,The Ministry of Health announced a new vaccina...,7,5,0.714286,False
2,Farmers should plant drought-resistant crops.,7,3,0.428571,True


## Exact-match flag

In [5]:
examples['exact_match'] = examples['reference'].str.strip() == examples['hypothesis'].str.strip()
examples[['source', 'exact_match', 'likely_omission']]

,source,exact_match,likely_omission
0,Wash your hands frequently with soap.,True,False
1,The Ministry of Health announced a new vaccina...,False,False
2,Farmers should plant drought-resistant crops.,False,True


## Error category summary
In a real run, extend this with: rare-word-containing sentences (`RareWordAccuracyEvaluator.split_by_rarity`), terminology mismatches (`TerminologyConsistencyChecker.check_translation`), and per-domain breakdowns joined from the source corpus's `Domain`/`source` column.

In [6]:
summary = {
    'total_examples': len(examples),
    'exact_matches': int(examples['exact_match'].sum()),
    'likely_omissions': int(examples['likely_omission'].sum()),
}
summary

{'total_examples': 3, 'exact_matches': 1, 'likely_omissions': 1}